In [ ]:
"""
Data Cleaning and Preprocessing Pipeline
Author: Hamid Rafique
Description: A robust data cleaning script to handle structural anomalies,
             missing data, and type normalization across retail datasets.
"""

import os
import pandas as pd


def load_datasets():
    """Loads CSV files using relative paths for cross-environment compatibility."""
    print("[INFO] Initializing file paths...")

    # Using relative paths so the code runs seamlessly on any machine
    data_dir = os.path.join(".", "data")

    prod_path = os.path.join(data_dir, "products.csv")
    ship_path = os.path.join(data_dir, "Shipping.csv")
    store_path = os.path.join(data_dir, "Stores.csv")
    trans_path = os.path.join(data_dir, "Transactions.csv")

    print("[INFO] Loading datasets into memory...")
    product_df = pd.read_csv(prod_path)
    shipping_df = pd.read_csv(ship_path)
    stores_df = pd.read_csv(store_path)
    transactions_df = pd.read_csv(trans_path)

    # Standardize headers to lowercase to eliminate syntax mismatches
    for df in [product_df, shipping_df, stores_df, transactions_df]:
        df.columns = df.columns.str.lower()

    return product_df, shipping_df, stores_df, transactions_df


def clean_products(product_df):
    """Cleans the products dataset: sequences IDs and extracts product features."""
    print("[INFO] Processing Products DataFrame...")

    # 1. Handle missing product IDs by position
    prod_id_col = product_df.columns[0]
    product_df[prod_id_col] = product_df[prod_id_col].astype(object)

    number_of_blanks = product_df[prod_id_col].isna().sum()
    if number_of_blanks > 0:
        new_ids = list(range(1, number_of_blanks + 1))
        blank_positions = product_df[prod_id_col].isna()
        product_df.loc[blank_positions, prod_id_col] = new_ids

    # 2. Extract Type and Colour fields from Product Name string split
    prod_name_col = product_df.columns[1]
    split_data = product_df[prod_name_col].str.split(" ", expand=True)

    product_df["type"] = split_data[0]
    if 1 in split_data.columns:
        product_df["colour"] = split_data[1]
    else:
        product_df["colour"] = None

    # 3. Standardize text variations
    product_df["colour"] = product_df["colour"].replace("Gray", "Grey")
    return product_df


def clean_shipping(shipping_df):
    """Cleans the shipping dataset: handles missing entries and typecasting."""
    print("[INFO] Processing Shipping DataFrame...")
    ship_order_col = shipping_df.columns[0]
    ship_type_col = shipping_df.columns[1]

    # Impute missing values with business fallback domain rule
    shipping_df[ship_type_col] = shipping_df[ship_type_col].fillna("Expedited")

    # Enforce relational consistency by casting Order ID to integer
    shipping_df[ship_order_col] = shipping_df[ship_order_col].astype(int)
    return shipping_df


def clean_stores(stores_df):
    """Cleans the stores dataset by purging completely invalid records."""
    print("[INFO] Processing Stores DataFrame...")
    store_id_col = stores_df.columns[0]

    # Drop structural records containing invalid primary keys
    stores_df = stores_df.dropna(subset=[store_id_col])
    stores_df = stores_df.reset_index(drop=True)
    return stores_df


def clean_transactions(transactions_df):
    """Cleans transaction logs: drops duplicated logs and normalizes time components."""
    print("[INFO] Processing Transactions DataFrame...")

    # Remove duplicated data rows
    duplicate_count = transactions_df.duplicated().sum()
    if duplicate_count > 0:
        print(f"[REMOVAL] Dropped {duplicate_count} duplicated log records.")
        transactions_df = transactions_df.drop_duplicates()
        transactions_df = transactions_df.reset_index(drop=True)

    # Cast calendar values to string formats for discrete analysis
    for col in ["year", "month", "day"]:
        if col in transactions_df.columns:
            transactions_df[col] = transactions_df[col].astype(str)

    return transactions_df


def main():
    """Core pipeline execution framework."""
    print("=" * 60)
    print("STARTING ETL DATA CLEANING PIPELINE")
    print("=" * 60)

    # Run execution pipeline steps
    product_df, shipping_df, stores_df, transactions_df = load_datasets()

    product_df = clean_products(product_df)
    shipping_df = clean_shipping(shipping_df)
    stores_df = clean_stores(stores_df)
    transactions_df = clean_transactions(transactions_df)

    print("\n" + "=" * 60)
    print("PIPELINE EXECUTED SUCCESSFULLY")
    print("=" * 60)


if __name__ == "__main__":
    main()

--- Initial Null Values Count ---
Products:
 productid    0
name         0
price        0
cost         0
dtype: int64 

Shipping:
 orderid           0
city              0
shippingmethod    9
dtype: int64 

Stores:
 storeid    10
city        0
dtype: int64 

Transactions:
 productid    0
year         0
month        0
day          0
quantity     0
orderline    0
orderid      0
dtype: int64 

--------------------------------------------------
Number of duplicate transaction rows removed: 7

VERIFYING FINAL RESULTS

--- Cleaned Stores Sample ---
Empty DataFrame
Columns: [storeid, city]
Index: []

--- Cleaned Products Sample ---
  productid           name  price  cost           type colour
0         1   Sweater,Blue     30    10   Sweater,Blue   None
1         2  Sweater,Black     35    10  Sweater,Black   None
2         3  Sweater,White     30    10  Sweater,White   None
3         4   Hoodie,Black     40    15   Hoodie,Black   None
4         5   Hoodie,White     35    15   Hoodie,White   N